# CRISP-DM: Donor Churn Prediction Pipeline

Goal: predict the likelihood that each donor will stop donating (no donation within 365 days).

## 1) Business & Data Understanding (Imports + Config)

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.db import load_env, build_engine
from src.features import build_donor_training_frame, split_feature_target, select_explanatory_features
from src.modeling import train_and_evaluate, score_donors, train_explanatory_model
from src.config import RANDOM_STATE, CHURN_WINDOW_DAYS, CV_FOLDS

pd.set_option('display.max_columns', 200)
start_all = time.time()

## Business KPI Mapping

- Stakeholder owner: Donor Operations Lead
- Decision enabled: prioritize retention outreach by donor risk tier
- Primary KPI: 90-day donor retention rate among contacted high-risk cohort
- Guardrail KPIs: outreach cost per retained donor, donor opt-out rate
- Minimum success criteria: +8% retention lift with <=15% increase in outreach cost

## Azure Connection Quick Test

If you want to test Azure directly, set `DB_PROFILE=azure` in `.env` (or load values from `.env.azure.example`) and run the next cell.

In [ ]:
# Optional: force Azure profile for this session
# os.environ['DB_PROFILE'] = 'azure'

from sqlalchemy import text

load_env('.env')
engine_azure = build_engine(os.getenv('DB_PROFILE', 'azure'))

with engine_azure.connect() as conn:
    print('Azure DB time:', conn.execute(text('select now();')).scalar())
    print('Current database:', conn.execute(text('select current_database();')).scalar())

## 2) Data Understanding (Connect + Snapshot Dataset)

In [ ]:
load_env('.env')
profile = os.getenv('DB_PROFILE', 'local').lower()
engine = build_engine(profile)
print(f'Connected with profile: {profile}')

t0 = time.time()
model_df = build_donor_training_frame(engine, churn_window_days=CHURN_WINDOW_DAYS)
print('Rows:', len(model_df), '| Donors:', model_df['supporter_id'].nunique())
print('Load+feature time (sec):', round(time.time() - t0, 2))
model_df.head()

In [ ]:
# Data Understanding Audit: missingness + simple anomaly checks
missing_pct = (model_df.isna().mean() * 100).sort_values(ascending=False)
print('Top missingness columns (%)')
display(missing_pct.head(10).to_frame('missing_pct'))

audit = {
    'rows': len(model_df),
    'distinct_donors': int(model_df['supporter_id'].nunique()),
    'negative_recency_rows': int((model_df['recency_days'] < 0).sum()) if 'recency_days' in model_df.columns else 0,
    'zero_or_negative_amount_avg_rows': int((model_df['amount_avg_to_date'] <= 0).sum()) if 'amount_avg_to_date' in model_df.columns else 0,
}
print('Audit summary:', audit)

In [ ]:
print('Target distribution (churn_365):')
display(model_df['churn_365'].value_counts(dropna=False).rename('count').to_frame())

display(model_df.describe(include='all').transpose().head(20))

## 3) Data Preparation (Feature/Target Split)

In [ ]:
X, y = split_feature_target(model_df)
print('X shape:', X.shape)
print('y mean (churn rate):', round(y.mean(), 4))
print('Nulls in X:', int(X.isna().sum().sum()))
X.head()

## 4) Modeling (Bounded Runtime)

Guardrails:
- only 2 models (Logistic Regression and Random Forest)
- capped CV folds (`CV_FOLDS`)
- bounded tree count
- elapsed time tracking

In [ ]:
t1 = time.time()
train_out = train_and_evaluate(X, y, random_state=RANDOM_STATE, cv_folds=CV_FOLDS)
print('Training time (sec):', round(time.time() - t1, 2))
display(train_out['results_df'])
print('Best model:', train_out['best_model_name'])

## 5) Explanatory Objective (Interpret Relationships, Not Causal Proof)

This section complements predictive modeling with an interpretable logistic specification.

- Goal: estimate direction and magnitude of relationships with churn.
- Caution: these are associations, not causal claims.
- Decision value: identify practical retention levers to test.

In [ ]:
X_expl = select_explanatory_features(X)
expl_out = train_explanatory_model(X_expl, y, random_state=RANDOM_STATE)

print('Explanatory model holdout metrics:')
print({k: round(v, 4) for k, v in expl_out['metrics'].items()})

print('\nTop positive churn associations (higher odds of churn):')
display(expl_out['top_positive'])

print('Top negative churn associations (lower odds of churn):')
display(expl_out['top_negative'])

### Decision Implications

Use the explanatory drivers as hypothesis generators for interventions:
- If higher recency (longer gap since prior donation) is associated with churn, trigger outreach earlier.
- If recurring behavior lowers churn odds, prioritize recurring conversion campaigns.
- If specific acquisition channels have higher churn association, redesign onboarding for those channels.

These actions should be validated with experiments (A/B tests), not assumed causal from this model alone.

## 6) Evaluation + Deployment-Style Scoring

In [ ]:
best_model = train_out['best_model']
scored = score_donors(best_model, X)

print('Risk tier counts:')
display(scored['risk_tier'].value_counts(dropna=False).to_frame('count'))
display(scored.head(20))

In [ ]:
if train_out['best_model_name'] == 'random_forest':
    prep = best_model.named_steps['prep']
    model = best_model.named_steps['model']
    feat_names = prep.get_feature_names_out()
    importances = pd.Series(model.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)
    display(importances.to_frame('importance'))
else:
    print('Top importances shown for random forest model only.')

## Operationalization Policy + Monitoring

- Threshold policy: use cost-minimizing threshold from confusion-cost analysis for campaign execution; default fallback 0.50.
- Action bands: `High` >= 0.65, `Medium` 0.35-0.65, `Low` < 0.35.
- Retraining cadence: monthly scheduled retrain; emergency retrain if PR-AUC drops >15% or key feature drift exceeds threshold.
- Integration references:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## 7) Conclusion

In [ ]:
print('Pipeline finished successfully.')
print('Total elapsed (sec):', round(time.time() - start_all, 2))